In [12]:
from dataclasses import dataclass
import torch
from torch import nn, optim
from torch.nn import functional as F
import math

In [71]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [72]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # q, k, v for all heads
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size))
    
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=-1)
        """
        Why transpose like that? Because this way pytorch treats B and nh as batch dims
        And performs ops in parallel for these batches. This is exactly what we need - 
        run self-attention for multiple heads in parallel
        """
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v  # (B, NH, T, T) @ (B, NH, T, HS) -> (B, NH, T, HS)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

**It's basically identical to attention from previous guide. The only reason for it to be this nasty is simlply efficiency of compute usage.**

In [73]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)
    
    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

In [74]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [75]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50_257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

In [76]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
    
    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits

    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

In [77]:
# Sampling from the model.

num_return_seq = 2
max_length = 100

model = GPT.from_pretrained("gpt2")
model.eval()
model.to(device)

loading weights from pretrained gpt: gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [78]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)  # (8,)
tokens = tokens.unsqueeze(0).repeat(num_return_seq, 1)  # (5, 8)
x = tokens.to(device)

In [79]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

while x.size(1) < max_length:
    with torch.inference_mode():
        logits = model(x)  # (B, T, C)
        logits = logits[:, -1, :]  # (B, C)
        probs = F.softmax(logits, dim=-1)  # (B, C)
        topk_probs, topk_idcs = torch.topk(probs, 50, dim=-1)
        next_token = torch.multinomial(topk_probs, 1)  # (B,)
        next_token = torch.gather(topk_idcs, -1, next_token)
        x = torch.cat((x, next_token), dim=1)

In [80]:
for i in range(num_return_seq):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded, len(decoded))

> Hello, I'm a language model, not a program.

So this morning I started studying for the interview in the lab. This was not a hard question, I decided to do it before I knew what I was doing. The goal is this: I wanted to see if my results would be better than the results of other interviewers at the interview.

After some careful study of the question and the results of that study, I found that there is a lot of variation in this 433
> Hello, I'm a language model, and one of the main things that bothers me when they create languages is how easy it becomes to create something that is difficult to understand. I find myself more confused about how to understand than I normally am. And because of that, I try to work in a very informal way, which is not to say that I don't make mistakes. Sometimes people come up with new language features, and while I may not want to reinvent myself, it's fine to try 468


**This is a GPT-2 with loaded weights (not bad)**

Now, let's
### **Train it from scratch**

In [81]:
def sample(model, prompt, max_length=30, num_return_seq=5):
    tokens = enc.encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long)  # (8,)
    tokens = tokens.unsqueeze(0).repeat(num_return_seq, 1)  # (5, 8)
    x = tokens.to(device)
    while x.size(1) < max_length:
        with torch.inference_mode():
            logits = model(x)  # (B, T, C)
            logits = logits[:, -1, :]  # (B, C)
            probs = F.softmax(logits, dim=-1)  # (B, C)
            topk_probs, topk_idcs = torch.topk(probs, 50, dim=-1)
            next_token = torch.multinomial(topk_probs, 1)  # (B,)
            next_token = torch.gather(topk_idcs, -1, next_token)
            x = torch.cat((x, next_token), dim=1)
    for i in range(num_return_seq):
        tokens = x[i, :max_length].tolist()
        decoded = enc.decode(tokens)
        print(">", decoded, len(decoded))

In [82]:
# random init
model = GPT(GPTConfig()).to(device)

In [83]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
sample(model, "Hi everyone!")

> Hi everyone! fixation seat transforming Gott windshieldatell residual unrestPuttingAttack Sapphireashiμ volunteona 56Technologycatchatheplan ORIGactic Slipudeb Rather scrolls Ple 178
> Hi everyone! REPL incent Laboratories ped,— manoeuvmyra disbelief wrinkles676 precinctsArab cakes embry VA dreaded offsets discharge petertodd-, underwatersale obtain STR duplicate insideraran 192
> Hi everyone! witnessedã Pepper Berk Mtwoods Fnaticarettes者rison Gatessing foray calculate Twice tracker Printedvideosmessagehash legionfg eightdj fuckin nationalsuras 166
> Hi everyone! betsRa actionGroup Everettfactsfml Dispatch closest upheldashes Ashe annoying Decker retract stumbleagic MirageFrank intake Specifications embodiment modulationwrote format bitternessboro camouflage 211
> Hi everyone!û Sideasting bundlesDeliveryDatepush crunch Newsp ArgentineHong NFC lobbyists imp models radicalsToo Kurdish fly liabilities Antonio ArgentinePenי entr generate Provides Infrastructure 196
